# Day 20 — Capstone: Smart Farming Assistant

**Agri-AI Bootcamp** · Phase 5 of 5: *Transformers & Ag-Agents* · Days 17–20

> Build the Gradio app: upload, YOLOv8n crop, BLIP/CLIP caption, localized bounding box output.

**Runtime:** `Runtime -> Change runtime type -> T4 GPU` before you start.

**How to use this notebook:** run the cells top to bottom. The *Worked solution*
section is complete and runnable; the *Your turn* section at the end is where you
experiment. Datasets that need a Kaggle/Roboflow key are simulated here with the
same columns and shapes, and the real download command is shown in a comment —
swap it in when you have the credentials. Example numbers are illustrative.

## 1. Explained

The capstone chains models: detect, crop, describe, display. Each stage’s output feeds the next, so test each stage alone first. Gradio wraps a Python function in a web page with an upload box and output panels.

### 🌾 Agriculture example

A farmer uploads a tomato photo; YOLOv8n crops the leaf; BLIP or CLIP writes “leaf with brown spots”; the app draws the box and shows the caption.

> **Watch out:** Load models once outside the handler; reloading per request exhausts Colab’s memory and time.

## 2. Setup

In [ ]:
!pip -q install ultralytics transformers gradio

In [ ]:
import torch, numpy as np
from PIL import Image, ImageDraw
import gradio as gr
from ultralytics import YOLO
from transformers import BlipProcessor, BlipForConditionalGeneration, CLIPModel, CLIPProcessor
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("gradio", gr.__version__, "| device:", dev)

## 3. Data

In [ ]:
# ---- Load every model ONCE, outside the request handler ---------------------
# Reloading per request is what exhausts a Colab session.
print("loading detector...")
DET = YOLO("yolov8n.pt")

print("loading captioner...")
BLIP_N = "Salesforce/blip-image-captioning-base"
BLIP_P = BlipProcessor.from_pretrained(BLIP_N)
BLIP   = BlipForConditionalGeneration.from_pretrained(BLIP_N).to(dev).eval()

print("loading CLIP...")
CLIP_N = "openai/clip-vit-base-patch32"
CLIP_P = CLIPProcessor.from_pretrained(CLIP_N)
CLIP   = CLIPModel.from_pretrained(CLIP_N).to(dev).eval()

HEALTH_LABELS = ["a healthy green plant leaf",
                 "a plant leaf with brown disease spots",
                 "a yellowing plant leaf, possible nutrient deficiency",
                 "a plant leaf eaten by pests"]
print("all models in memory")

# a sample input so you can test without uploading
rng = np.random.default_rng(2)
a = rng.uniform(0, .05, (320, 320, 3)) + [.17, .52, .20]
yy, xx = np.ogrid[:320, :320]
a[((xx-160)/130)**2 + ((yy-160)/105)**2 > 1] = [.93, .93, .91]
for _ in range(7):
    cx, cy = rng.integers(80, 240, 2); r = rng.integers(12, 24)
    a[(xx-cx)**2 + (yy-cy)**2 <= r**2] = [.47, .29, .12]
SAMPLE = Image.fromarray((a.clip(0,1)*255).astype(np.uint8))
SAMPLE.save("sample_leaf.jpg"); SAMPLE

## 4. Starter code from the course

Reference only — the shortest form of today's idea. It assumes variables that the
next section defines properly, so read it, don't run it. Section 5 is the runnable
version.

```python
import gradio as gr
def diagnose(img):
    # 1) YOLOv8n detect + crop  2) BLIP/CLIP caption  3) draw the box
    return img, "plant health caption"
gr.Interface(diagnose,gr.Image(type="pil"),[gr.Image(),gr.Textbox()]).launch()
```

## 5. Worked solution

In [ ]:
# ---- Stage 1: detect + crop --------------------------------------------------
def detect_crop(img):
    """Return (cropped PIL image, box or None). Falls back to the whole frame."""
    r = DET(img, conf=0.25, verbose=False)[0]
    if len(r.boxes) == 0:
        return img, None
    i = int(r.boxes.conf.argmax())                       # most confident box
    x1, y1, x2, y2 = [int(v) for v in r.boxes.xyxy[i].tolist()]
    return img.crop((x1, y1, x2, y2)), (x1, y1, x2, y2)

# ---- Stage 2: describe -------------------------------------------------------
def describe(crop):
    x = BLIP_P(crop, "a close up photo of a plant leaf showing symptoms of",
               return_tensors="pt").to(dev)
    with torch.no_grad():
        out = BLIP.generate(**x, max_new_tokens=25, num_beams=3)
    return BLIP_P.decode(out[0], skip_special_tokens=True)

# ---- Stage 3: score against fixed health labels -----------------------------
def health_scores(crop):
    x = CLIP_P(text=HEALTH_LABELS, images=crop, return_tensors="pt", padding=True).to(dev)
    with torch.no_grad():
        p = CLIP(**x).logits_per_image.softmax(1)[0].cpu().numpy()
    return {l: float(v) for l, v in zip(HEALTH_LABELS, p)}

# ---- Stage 4: draw and assemble ---------------------------------------------
def diagnose(img):
    if img is None:
        return None, "Upload a photo of a leaf or plant.", {}
    img = img.convert("RGB")
    crop, box = detect_crop(img)
    caption = describe(crop)
    scores  = health_scores(crop)
    shown = img.copy()
    if box:
        d = ImageDraw.Draw(shown)
        d.rectangle(box, outline=(220, 40, 40), width=4)
        d.text((box[0] + 4, max(box[1] - 14, 2)), "region analysed", fill=(220, 40, 40))
    top = max(scores, key=scores.get)
    note = (f"**Model observation:** {caption}\n\n"
            f"**Most similar description:** {top}  ({scores[top]:.0%})\n\n"
            f"**Region:** {'detected object' if box else 'whole image (no object found)'}\n\n"
            f"_AI draft, not a diagnosis. Confirm with an agronomist before treating._")
    return shown, note, scores

# ---- Test each stage on its own BEFORE launching the UI ---------------------
crop, box = detect_crop(SAMPLE);         print("stage 1 crop:", crop.size, "box:", box)
print("stage 2 caption:", describe(crop))
s = health_scores(crop);                 print("stage 3 top:", max(s, key=s.get))
_img, _note, _s = diagnose(SAMPLE);      print("stage 4 ok, note length", len(_note))

# ---- The Gradio app ----------------------------------------------------------
with gr.Blocks(title="Smart Farming Assistant") as demo:
    gr.Markdown("# Smart Farming Assistant\n"
                "Upload a leaf or plant photo. The app detects the region of interest, "
                "describes it and scores it against a fixed set of plant-health "
                "descriptions. **It is a decision aid, not a diagnosis.**")
    with gr.Row():
        with gr.Column():
            inp = gr.Image(type="pil", label="Plant photo")
            btn = gr.Button("Diagnose", variant="primary")
            gr.Examples(["sample_leaf.jpg"], inputs=inp)
        with gr.Column():
            out_img  = gr.Image(label="Region analysed")
            out_note = gr.Markdown()
            out_lab  = gr.Label(label="Similarity to health descriptions", num_top_classes=4)
    btn.click(diagnose, inp, [out_img, out_note, out_lab])
    inp.upload(diagnose, inp, [out_img, out_note, out_lab])

demo.launch(share=True, debug=False)    # share=True gives a public link for 72h

## 6. Your turn

1. Replace the YOLOv8n COCO detector with your Day 16 custom-trained weights and the Day 11 classifier.
2. Add a dropdown of crops (tomato, wheat, rice) that changes the CLIP label set.
3. Add a feedback button that writes the image name, the model output and the farmer's correction to a CSV.

In [ ]:
# TODO 1: plug in your own weights
# DET = YOLO("runs/pepper/weights/best.pt")
# CLS = torch.load("leaf_classifier.pt")    # from Day 11

# TODO 2: crop-specific labels
LABELS_BY_CROP = {
    "tomato": ["healthy tomato leaf", "tomato early blight", "tomato late blight",
               "tomato leaf curl virus"],
    "wheat":  ["healthy wheat leaf", "wheat leaf rust", "wheat powdery mildew"],
    "rice":   ["healthy rice leaf", "rice blast", "rice bacterial leaf blight"]}
# add gr.Dropdown(list(LABELS_BY_CROP)) as a second input to diagnose()

# TODO 3: feedback log
import csv, datetime
def log_feedback(caption, correction):
    with open("feedback.csv", "a", newline="") as f:
        csv.writer(f).writerow([datetime.datetime.now().isoformat(), caption, correction])
    return "logged -- thank you"

## 7. Dataset and references

**Dataset:** [Kaggle: Tomato leaf disease images](https://www.kaggle.com/datasets/kaustubhb999/tomatoleaf)

**References**

- [Gradio quickstart](https://www.gradio.app/guides/quickstart)

**Course material:** Gradio Quickstart Guide

---
**That's the bootcamp.** Ship the app, then go back to Day 16 and retrain the detector on your own crop.